In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# KINGElvis FileName
file_name="VTA_CA_OB_KINGElvis.xlsx"

if file_name.split('_')[0].isdigit():
    file_first_name=file_name.split('_')[0]+'_'+file_name.split('_')[1]
else:
    file_first_name=file_name.split('_')[0]

# in some Compeletion Report LSNAMECODE is splited in some it is not so have to check that
def edit_ls_code_column(x):
    value=x.split('_')
    if len(value)>3:
        route_value="_".join(value[:-1])
    else:
        route_value="_".join(value)
    return route_value

# for generated file version
version=2
project_name='TUCSON'
today_date = date.today()
today_date=''.join(str(today_date).split('-'))

detail_df_stops=pd.read_excel('details_project_od_MUNI.xlsx',sheet_name='STOPS')
detail_df_xfers=pd.read_excel('details_project_od_MUNI.xlsx',sheet_name='XFERS')

wkend_overall_df=pd.read_excel('project_CR_MUNI.xlsx',sheet_name='WkEND-Overall')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('project_CR_MUNI.xlsx',sheet_name='WkEND-RouteTotal')
df=pd.read_csv("reviewtool_20250404_MUNI_ROUTE_DIRECTION_CHECk.csv")

# if we have generated route_direction_database file using route_direction_refator_database.py file then have to replace and rename the columns
df.drop(columns=['ROUTE_SURVEYEDCode','ROUTE_SURVEYED'],inplace=True)
df.rename(columns={'ROUTE_SURVEYEDCode_New':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_NEW':'ROUTE_SURVEYED'},inplace=True) 

wkday_overall_df=pd.read_excel('project_CR_MUNI.xlsx',sheet_name='WkDAY-Overall')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('project_CR_MUNI.xlsx',sheet_name='WkDAY-RouteTotal')

df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )

# Getting Data from Database where the Final Usage is Use in KINGELVIS  
# df=pd.merge(df,ke_df['id'],on='id',how='inner')
df=df[df['INTERV_INIT']!='999']
df=df[df['INTERV_INIT']!=999]
df.drop_duplicates(subset='id',inplace=True)


def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

In [2]:
elvis_status_check=['elvisstatus']
elvis_status=check_all_characters_present(df,elvis_status_check)
elvis_status

['ELVIS_STATUS']

In [5]:
df=df[df[elvis_status[0]].str.lower()!='delete']

In [6]:
df[df[elvis_status[0]]=='delete']

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,STOP_ON_LONG_NEW,STOP_OFF_ADDRESS_NEW,STOP_OFF_SEQ,STOP_OFF_CLINTID_NEW,STOP_OFF_LAT_NEW,STOP_OFF_LONG_NEW,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,STOP_ON_ADDRESS_NEW,ROUTE_SURVEYEDCode_Splited


In [2]:
def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)

In [3]:
def get_day_name(x):
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue

df['Day']=df['Date'].apply(get_day_name)


In [4]:
df['LAST_SURVEY_DATE'] = pd.to_datetime(df['Date'], format='%Y-%m-%d %H:%M:%S')
latest_date = df['LAST_SURVEY_DATE'].max()
latest_date_df = pd.DataFrame({'Latest_Survey_Date': [latest_date]})


weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]

weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]


In [5]:
time_column_check=['timeoncode']
time_period_column_check=['timeon']
time_column=check_all_characters_present(df,time_column_check)
time_period_column=check_all_characters_present(df,time_period_column_check)
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)
stopon_clntid_column_check=['stoponclntid']
stopon_clntid_column=check_all_characters_present(df,stopon_clntid_column_check)
stopoff_clntid_column_check=['stopoffclntid']
stopoff_clntid_column=check_all_characters_present(df,stopoff_clntid_column_check)


In [6]:

wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)
wkday_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)


In [7]:
# for col in [0, 1, 2, 3, 4, 5, 6]:
#     wkday_overall_df[col] = pd.to_numeric(wkday_overall_df[col], errors='coerce')

In [8]:
# for col in [0, 1, 2, 3, 4, 5, 6]:
#     wkend_overall_df[col] = pd.to_numeric(wkend_overall_df[col], errors='coerce')

In [9]:
# wkend_overall_df[[0,1,2,3,4,5,6]].fillna(0,inplace=True)

In [10]:
# wkday_overall_df[[0,1,2,3,4,5,6]].fillna(0,inplace=True)

In [11]:
def create_time_value_df_with_display(df):
    """
    Create a time-value DataFrame summarizing counts and time ranges.

    Parameters:
        df (pd.DataFrame): Input DataFrame containing the time values.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # pre_early_am_values = ['AM1']
    early_am_values = ['AM1','AM2']
    am_values = ['AM3', 'MID1','MID2']
    midday_values = [ 'MID3', 'MID4', 'MID5', 'MID6','MID7' ]
    pm_values = ['PM1','PM2']
    pm_peak_values = ['PM3','PM4','PM5']
    evening_values = ['PM6','PM7','PM8']
    night_values = ['PM9']

    # Mapping time groups to corresponding columns
    time_group_mapping = {
        # 0: pre_early_am_values,
        0: early_am_values,
        1: am_values,
        2: midday_values,
        3: pm_values,
        4: pm_peak_values,
        5: evening_values,
        6: night_values
    }
    time_mapping = {
    'AM1': 'Before 5:00 am',
    'AM2': '5:00 am - 6:00 am',
    'AM3': '6:00 am - 7:00 am',
    'MID1': '7:00 am - 8:00 am',
    'MID2': '8:00 am - 9:00 am',
    'MID7': '9:00 am - 10:00 am',
    'MID3': '10:00 am - 11:00 am',
    'MID4': '11:00 am - 12:00 pm',
    'MID5': '12:00 pm - 1:00 pm',
    'MID6': '1:00 pm - 2:00 pm',
    'PM1': '2:00 pm - 3:00 pm',
    'PM2': '3:00 pm - 4:00 pm',
    'PM3': '4:00 pm - 5:00 pm',
    'PM4': '5:00 pm - 6:00 pm',
    'PM5': '6:00 pm - 7:00 pm',
    'PM6': '7:00 pm - 8:00 pm',
    'PM7': '8:00 pm - 9:00 pm',
    'PM8': '9:00 pm - 10:00 pm',
    'PM9': 'After 10:00 pm'
}


    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text", 0, 1, 2, 3, 4,5,6])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = df[df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df


In [12]:
wkend_time_value_df=create_time_value_df_with_display(weekend_df)
wkday_time_value_df=create_time_value_df_with_display(weekday_df)

In [13]:
wkend_time_value_df.head()

,Original Text,0,1,2,3,4,5,6,Time Range,Display_Text
0,AM1,2,0,0,0,0,0,NaN,Before 5:00 am,1
1,AM2,48,0,0,0,0,0,NaN,5:00 am - 6:00 am,2
2,AM3,0,75,0,0,0,0,NaN,6:00 am - 7:00 am,3
3,MID1,0,123,0,0,0,0,NaN,7:00 am - 8:00 am,4
4,MID2,0,193,0,0,0,0,NaN,8:00 am - 9:00 am,5


In [14]:
wkday_time_value_df.head()

,Original Text,0,1,2,3,4,5,6,Time Range,Display_Text
0,AM1,71,0,0,0,0,0,NaN,Before 5:00 am,1
1,AM2,219,0,0,0,0,0,NaN,5:00 am - 6:00 am,2
2,AM3,0,1047,0,0,0,0,NaN,6:00 am - 7:00 am,3
3,MID1,0,1717,0,0,0,0,NaN,7:00 am - 8:00 am,4
4,MID2,0,1728,0,0,0,0,NaN,8:00 am - 9:00 am,5


In [15]:

# To create Route_SurveyedCode Direction wise comparison in terms of time values
def create_route_direction_level_df(overalldf,df):
    early_am_values = ['AM1','AM2']
    am_values = ['AM3', 'MID1','MID2']
    midday_values = [ 'MID3', 'MID4', 'MID5', 'MID6','MID7' ]
    pm_values = ['PM1','PM2']
    pm_peak_values = ['PM3','PM4','PM5']
    evening_values = ['PM6','PM7','PM8']
    night_values = ['PM9']

    early_am_column = [0]
    am_column=[1] #This is for AM header
    midday_colum=[2] #this is for MIDDAY header
    pm_column=[3] #this is for PM header
    pm_peak_column=[4] #this is for PM header
    evening_column=[5] #this is for EVENING header
    night_column=[6] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0
        
    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report

    # For TUCSON PROJECT we are not using pre_early_am and early_am columns so have to comment the following code accordingly 
    # print(early_am_column[0])

    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    # new_df['CR_PRE_Early_AM']=overalldf[pre_early_am_column[0]].apply(math.ceil)
    new_df['CR_Early_AM'] =  pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] =  pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] =  pd.to_numeric(overalldf[midday_colum[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM'] =  pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] =  pd.to_numeric(overalldf[pm_peak_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] =  pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_NIGHT'] =  pd.to_numeric(overalldf[night_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    # new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_NIGHT']]=new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_NIGHT']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']

        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        # pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        pm_peak_value, pm_peak_value_ids = get_counts_and_ids(pm_peak_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        night_value, night_value_ids = get_counts_and_ids(night_values)
        
        # new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_Total'] =row['CR_EARLY_AM']+ row['CR_AM_Peak'] + row['CR_Midday']  + row['CR_PM'] + row['CR_PM_Peak'] + row['CR_Evening']+ row['CR_NIGHT']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']

        # new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM'] = pm_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_peak_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Night'] = night_value
        # new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        new_df.loc[index, 'DB_Total'] = evening_value + early_am_value + am_value + midday_value + pm_value + pm_peak_value + night_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )
        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            # pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_diff=row['CR_PM']-row['DB_PM']
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            night_diff=row['CR_Night']-row['DB_Night']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            # new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            # new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_diff))
            new_df.loc[index, 'PM_PEAK_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            new_df.loc[index, 'Night_DIFFERENCE'] = math.ceil(max(0, night_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            # new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, early_am_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, pm_diff))+math.ceil(max(0, evening_diff))+math.ceil(max(0, night_diff))

    return new_df


In [22]:

def create_route_direction_level_df(overalldf, df):
    early_am_values = ['AM1','AM2']
    am_values = ['AM3', 'MID1','MID2']
    midday_values = ['MID3', 'MID4', 'MID5', 'MID6','MID7']
    pm_values = ['PM1','PM2']
    pm_peak_values = ['PM3','PM4','PM5']
    evening_values = ['PM6','PM7','PM8']
    night_values = ['PM9']

    early_am_column = [0]
    am_column = [1]
    midday_colum = [2]
    pm_column = [3]
    pm_peak_column = [4]
    evening_column = [5]
    night_column = [6]

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode'] = overalldf['LS_NAME_CODE']
    
    # Create columns with consistent naming
    new_df['CR_Early_AM'] = pd.to_numeric(overalldf[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overalldf[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overalldf[midday_colum[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM'] = pd.to_numeric(overalldf[pm_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overalldf[pm_peak_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overalldf[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Night'] = pd.to_numeric(overalldf[night_column[0]], errors='coerce').fillna(0).apply(math.ceil)

    new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_Night']] = \
        new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_Night']].applymap(convert_string_to_integer)
    new_df.fillna(0, inplace=True)

    # Define time_column (was missing in original code)
#     time_column = ['your_time_column_name']  # Replace with actual column name from df

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']

        def get_counts_and_ids(time_values):
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        pm_peak_value, pm_peak_value_ids = get_counts_and_ids(pm_peak_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        night_value, night_value_ids = get_counts_and_ids(night_values)
        
        # Use consistent column names (matching the creation above)
        new_df.loc[index, 'CR_Total'] = (row['CR_Early_AM'] + row['CR_AM_Peak'] + row['CR_Midday'] + 
                                        row['CR_PM'] + row['CR_PM_Peak'] + row['CR_Evening'] + row['CR_Night'])
        new_df.loc[index, 'CR_AM_Peak'] = row['CR_AM_Peak']

        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM'] = pm_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_peak_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Night'] = night_value
        new_df.loc[index, 'DB_Total'] = (evening_value + early_am_value + am_value + midday_value + 
                                        pm_value + pm_peak_value + night_value)

        route_code_level_df=pd.DataFrame()

        unique_routes=new_df['ROUTE_SURVEYEDCode'].unique()

        route_code_level_df['ROUTE_SURVEYEDCode']=unique_routes

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            # pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            early_am_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_diff=row['CR_PM']-row['DB_PM']
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            night_diff=row['CR_Night']-row['DB_Night']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            # new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            # new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_diff))
            new_df.loc[index, 'PM_PEAK_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            new_df.loc[index, 'Night_DIFFERENCE'] = math.ceil(max(0, night_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            # new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, early_am_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, pm_diff))+math.ceil(max(0, evening_diff))+math.ceil(max(0, night_diff))

    return new_df

# Usage
wkend_route_direction_df = create_route_direction_level_df(wkend_overall_df, weekend_df)
wkday_route_direction_df = create_route_direction_level_df(wkday_overall_df, weekday_df)

In [30]:
wkday_route_direction_df.to_csv('WkDAY ROUTE DIRECTION.csv',index=False)

In [28]:
import pandas as pd
import math

def create_route_level_df(overall_df, route_df, df):
    early_am_values = ['AM1','AM2']
    am_values = ['AM3', 'MID1','MID2']
    midday_values = ['MID3', 'MID4', 'MID5', 'MID6','MID7']
    pm_values = ['PM1','PM2']
    pm_peak_values = ['PM3','PM4','PM5']
    evening_values = ['PM6','PM7','PM8']
    night_values = ['PM9']

    early_am_column = [0]
    am_column = [1]
    midday_colum = [2]
    pm_column = [3]
    pm_peak_column = [4]
    evening_column = [5]
    night_column = [6]

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode'] = overall_df['LS_NAME_CODE']
    
    # Create columns with consistent naming
    new_df['CR_Early_AM'] = pd.to_numeric(overall_df[early_am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_AM_Peak'] = pd.to_numeric(overall_df[am_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Midday'] = pd.to_numeric(overall_df[midday_colum[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM'] = pd.to_numeric(overall_df[midday_colum[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_PM_Peak'] = pd.to_numeric(overall_df[pm_peak_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Evening'] = pd.to_numeric(overall_df[evening_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    new_df['CR_Night'] = pd.to_numeric(overall_df[night_column[0]], errors='coerce').fillna(0).apply(math.ceil)
    
    new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_Night']] = \
        new_df[['CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM','CR_PM_Peak','CR_Evening','CR_Night']].applymap(convert_string_to_integer)
    new_df.fillna(0, inplace=True)

    # Define time_column - replace with your actual column name
#     time_column = ['Time_Period']  # Adjust this to match your df column name containing AM1, AM2, etc.

    # Populate DB columns
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']

        def get_counts_and_ids(time_values):
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids
        
        early_am_value, _ = get_counts_and_ids(early_am_values)
        am_value, _ = get_counts_and_ids(am_values)
        midday_value, _ = get_counts_and_ids(midday_values)
        pm_value, _ = get_counts_and_ids(pm_values)
        pm_peak_value, _ = get_counts_and_ids(pm_peak_values)
        evening_value, _ = get_counts_and_ids(evening_values)
        night_value, _ = get_counts_and_ids(night_values)
        
        # Use consistent column names
        new_df.loc[index, 'CR_Total'] = (row['CR_Early_AM'] + row['CR_AM_Peak'] + row['CR_Midday'] + 
                                        row['CR_PM'] + row['CR_PM_Peak'] + row['CR_Evening'] + row['CR_Night'])
        new_df.loc[index, 'CR_AM_Peak'] = row['CR_AM_Peak']

        new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM'] = pm_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_peak_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Night'] = night_value
        new_df.loc[index, 'DB_Total'] = (evening_value + early_am_value + am_value + midday_value + 
                                        pm_value + pm_peak_value + night_value)

    # Route Level Comparison
    new_df['ROUTE_SURVEYEDCode_Splited'] = new_df['ROUTE_SURVEYEDCode'].apply(lambda x: '_'.join(x.split('_')[:-1]))
    new_df.to_csv('W')
    # Create route_level_df
    route_level_df = pd.DataFrame()
    unique_routes = new_df['ROUTE_SURVEYEDCode_Splited'].unique()
    route_level_df['ROUTE_SURVEYEDCode'] = unique_routes

    route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'}, inplace=True)
    route_df.dropna(subset=['ROUTE_SURVEYEDCode'], inplace=True)
    route_level_df = pd.merge(route_level_df, route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']], on='ROUTE_SURVEYEDCode')

    # Populate route-level sums
    for index, row in route_level_df.iterrows():
        subset_df = new_df[new_df['ROUTE_SURVEYEDCode_Splited'] == row['ROUTE_SURVEYEDCode']]
        sum_per_route_cr = subset_df[['CR_Early_AM', 'CR_AM_Peak', 'CR_Midday', 'CR_PM', 'CR_PM_Peak', 'CR_Evening', 'CR_Night', 'CR_Total']].sum()
        sum_per_route_db = subset_df[['DB_Early_AM_Peak','DB_AM_Peak', 'DB_Midday', 'DB_PM', 'DB_PM_Peak', 'DB_Evening', 'DB_Night', 'DB_Total']].sum()
        
        route_level_df.loc[index,'CR_Early_AM'] = sum_per_route_cr['CR_Early_AM']
        route_level_df.loc[index,'CR_AM_Peak'] = sum_per_route_cr['CR_AM_Peak']
        route_level_df.loc[index,'CR_Midday'] = sum_per_route_cr['CR_Midday']
        route_level_df.loc[index,'CR_PM'] = sum_per_route_cr['CR_PM']
        route_level_df.loc[index,'CR_PM_Peak'] = sum_per_route_cr['CR_PM_Peak']
        route_level_df.loc[index,'CR_Evening'] = sum_per_route_cr['CR_Evening']
        route_level_df.loc[index,'CR_Night'] = sum_per_route_cr['CR_Night']
        route_level_df.loc[index,'CR_Total'] = sum_per_route_cr['CR_Total']
        
        route_level_df.loc[index,'DB_Early_AM_Peak'] = sum_per_route_db['DB_Early_AM_Peak']
        route_level_df.loc[index,'DB_AM_Peak'] = sum_per_route_db['DB_AM_Peak']
        route_level_df.loc[index,'DB_Midday'] = sum_per_route_db['DB_Midday']
        route_level_df.loc[index,'DB_PM'] = sum_per_route_db['DB_PM']
        route_level_df.loc[index,'DB_PM_Peak'] = sum_per_route_db['DB_PM_Peak']
        route_level_df.loc[index,'DB_Evening'] = sum_per_route_db['DB_Evening']
        route_level_df.loc[index,'DB_Night'] = sum_per_route_db['DB_Night']
        route_level_df.loc[index,'DB_Total'] = sum_per_route_db['DB_Total']

    # Calculate differences
    for index, row in route_level_df.iterrows():
        early_am_peak_diff = row['CR_Early_AM'] - row['DB_Early_AM_Peak']
        am_peak_diff = row['CR_AM_Peak'] - row['DB_AM_Peak']
        midday_diff = row['CR_Midday'] - row['DB_Midday']
        pm_diff = row['CR_PM'] - row['DB_PM']
        pm_peak_diff = row['CR_PM_Peak'] - row['DB_PM_Peak']
        evening_diff = row['CR_Evening'] - row['DB_Evening']
        night_diff = row['CR_Night'] - row['DB_Night']
        total_diff = row['CR_Total'] - row['DB_Total']
        overall_difference = row['CR_Overall_Goal'] - row['DB_Total']

        route_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_diff))
        route_level_df.loc[index, 'PM_PEAK_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        route_level_df.loc[index, 'Night_DIFFERENCE'] = math.ceil(max(0, night_diff))
        route_level_df.loc[index, 'Total_DIFFERENCE'] = (math.ceil(max(0, early_am_peak_diff)) + 
                                                        math.ceil(max(0, am_peak_diff)) + 
                                                        math.ceil(max(0, midday_diff)) + 
                                                        math.ceil(max(0, pm_diff)) + 
                                                        math.ceil(max(0, pm_peak_diff)) + 
                                                        math.ceil(max(0, evening_diff)) + 
                                                        math.ceil(max(0, night_diff)))
        route_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0, overall_difference))

    return route_level_df

# Usage
wkday_route_level =create_route_level_df(wkday_overall_df,wkday_route_df,weekday_df)

In [29]:
wkday_route_level.to_csv('wkday_route_level.csv',index=False)